# Lesson 2.4 — experiments: heuristics in the warehouse

Run the cells in order. Before each experiment, write down your prediction in the "Your prediction" cell (double-click it to edit), and only then run the code.

The first cell chooses the implementation:

- `"mine"` uses your `task.py`; solve Part B of the task first;
- `"reference"` uses the course's reference implementation. Set `IMPLEMENTATION = "reference"` if you want to run the notebook before finishing the task.

In [ ]:
%load_ext autoreload
%autoreload 2
import os

# "mine" uses your task.py; "reference" uses the course's reference implementation.
IMPLEMENTATION = os.environ.get("INTROAI_IMPLEMENTATION", "mine")
if IMPLEMENTATION == "mine":
    from task import a_star, greedy_best_first, manhattan
else:
    from introai.search.reference import a_star, greedy_best_first
    from introai.world.reference import manhattan

The function `run` runs one search on a map and returns the goal node and the list of expanded nodes. Searches with a heuristic get a function `h(cell)`, and the function `to_dock(warehouse, distance)` builds one from a distance to the dock. UCS always comes from the course's reference implementation (lesson 2.3). In the pictures blue cells are expanded, orange cells are the route, and green hatched cells are slow zones.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from introai.search.reference import ucs
from introai.trace import TraceLimitExceeded, record
from introai.world.plot import plot_map
from introai.world.warehouse import RouteProblem, WarehouseMap, render_text

MAPS = Path.cwd().parents[2] / "data" / "maps"


def to_dock(warehouse, distance):
    """A heuristic: the given distance from a cell to the dock."""
    return lambda cell: distance(cell, warehouse.dock)


def run(search, warehouse, h=None, limit=100_000):
    """Run one search on a map; return the goal node and the expanded nodes."""
    problem = RouteProblem(warehouse)
    try:
        with record(limit=limit, kinds={"expand"}) as trace:
            node = search(problem) if h is None else search(problem, h)
    except TraceLimitExceeded:
        raise RuntimeError(
            f"{search.__name__} expanded more than {limit} nodes - check task.py "
            '(or set IMPLEMENTATION = "reference" in the first cell)'
        ) from None
    return node, [event.data["node"] for event in trace.of_kind("expand")]


def summary(label, node, expanded):
    if node is None:
        print(f"{label}: no route, expanded {len(expanded)}")
    else:
        print(
            f"{label}: {len(node.solution())} steps, cost {node.path_cost:g}, "
            f"expanded {len(expanded)}"
        )

## Experiment 1: UCS, greedy and A* in a congested warehouse

The same map as in lesson 2.3. Greedy and A\* get the Manhattan distance to the dock.

In [ ]:
congested = WarehouseMap.from_file(MAPS / "congested.txt")
print(render_text(congested))

**Your prediction.** Which way will each of the three algorithms go? Which one will expand the most cells, and which the fewest? Will A\* look into the storage area on the left?

*My prediction:* …

In [ ]:
h = to_dock(congested, manhattan)
runs = [
    ("UCS", run(ucs, congested)),
    ("greedy best-first", run(greedy_best_first, congested, h)),
    ("A*", run(a_star, congested, h)),
]
fig, axes = plt.subplots(3, 1, figsize=(10, 9))
for ax, (label, (node, expanded)) in zip(axes, runs):
    summary(label, node, expanded)
    path = node.states() if node else []
    plot_map(congested, path=path, expanded=[n.state for n in expanded], title=label, ax=ax)
fig.tight_layout()

**Think about it.** Compare the result with your prediction. Why did greedy drive into the corridor, even though it is the most expensive route? Which cells of the storage area on the left did A\* expand, and why exactly those? Work out *f* = *g* + *h* for one of them and compare it with the cost of the route.

## Experiment 2: a weaker and a stronger heuristic

We compare A\* with five heuristics: *h* = 0, the Euclidean (straight-line) distance, the Manhattan distance, and the Manhattan distance multiplied by 2 and by 3.

**Your prediction.** Which of these heuristics are admissible? How will the number of expansions and the cost of the route change as the heuristic grows?

*My prediction:* …

In [ ]:
def euclidean(cell, goal):
    return ((cell[0] - goal[0]) ** 2 + (cell[1] - goal[1]) ** 2) ** 0.5


heuristics = {
    "h = 0": lambda cell: 0,
    "Euclidean": to_dock(congested, euclidean),
    "Manhattan": to_dock(congested, manhattan),
    "2 x Manhattan": lambda cell: 2 * manhattan(cell, congested.dock),
    "3 x Manhattan": lambda cell: 3 * manhattan(cell, congested.dock),
}
labels, expansions, costs = [], [], []
for label, h in heuristics.items():
    node, expanded = run(a_star, congested, h)
    summary(f"A* with {label}", node, expanded)
    labels.append(label)
    expansions.append(len(expanded))
    costs.append(node.path_cost)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, values, title in [
    (axes[0], expansions, "Cells expanded by A*"),
    (axes[1], costs, "Cost of the route A* finds"),
]:
    ax.bar(labels, values, color="#2a78d6", width=0.6)
    ax.set_title(title, loc="left", color="#0b0b0b")
    ax.grid(axis="y", color="#e1e0d9", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color("#c3c2b7")
    ax.tick_params(colors="#52514e")
    ax.tick_params(axis="x", labelrotation=20)
fig.tight_layout()

**Think about it.** Which of these heuristics are admissible? With which heuristic does A\* first return a more expensive route? Is the doubled Manhattan distance admissible, given that A\* still found the cheapest route with it on this map? The Euclidean distance expanded as many cells here as the Manhattan distance. In experiment 3 you will check whether that is always so.

## Experiment 3: aisles and the open floor

These maps have no slow zones, so the Manhattan distance is often very close to the true cost.

**Your prediction.** How many times fewer cells will A\* with the Manhattan distance expand than UCS on the map with aisles, and how many times on the open floor? Will A\* with the Euclidean distance expand as many cells?

*My prediction:* …

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(17, 7))
for row, name in enumerate(["aisles", "open_floor"]):
    warehouse = WarehouseMap.from_file(MAPS / f"{name}.txt")
    node, expanded = run(a_star, warehouse, to_dock(warehouse, euclidean))
    summary(f"{name}, A* with Euclidean", node, expanded)
    for col, (label, search, h) in enumerate(
        [("UCS", ucs, None), ("A* with Manhattan", a_star, to_dock(warehouse, manhattan))]
    ):
        node, expanded = run(search, warehouse, h)
        summary(f"{name}, {label}", node, expanded)
        path = node.states() if node else []
        expanded_cells = [n.state for n in expanded]
        title = f"{name}: {label}"
        plot_map(warehouse, path=path, expanded=expanded_cells, title=title, ax=axes[row][col])
fig.tight_layout()

**Think about it.** On which map does the heuristic help more, and why? When is the Manhattan distance equal to the true cost of the route, and when does it underestimate it? Compare the Euclidean and the Manhattan distance for one cell: which is larger? How does that translate into the number of expansions?

## Summary

Go back to the review questions in the lesson. After your own attempt, you can compare your results and conclusions with the answers in `solutions/02_search/04_heuristics_and_a_star/answers.md`.